# Assignment 9: Full Round 2-Style Problem — Neural Operator for PDEs (100 points)

## Problem Description

This problem combines PINNs, novel architectures, and paper-to-implementation skills. You will implement a **Fourier Neural Operator (FNO)**, which learns to map between function spaces — specifically, mapping initial conditions to PDE solutions.

### Background: From PINNs to Neural Operators

A PINN approximates the solution $u(t, x)$ for **one specific** PDE instance (one IC, one BC). If the IC changes, you must retrain.

A **Neural Operator** learns the mapping $\mathcal{G}: u_0 \mapsto u_T$ — given ANY initial condition $u_0(x)$, predict the solution at time $T$. Train once, evaluate for any IC.

### Fourier Neural Operator (FNO)

The key insight: convolutions in physical space become multiplications in Fourier space. FNO applies learnable transformations in Fourier space.

**FNO Layer:**

Given input $v \in \mathbb{R}^{B \times N \times d_v}$ (discretized function on $N$ grid points with $d_v$ channels):

1. **Fourier branch:** 
   - Apply FFT along the spatial dimension: $\hat{v} = \text{FFT}(v)$ → shape $(B, N, d_v)$ complex
   - Keep only the first $k_{\max}$ Fourier modes: $\hat{v}_{\text{trunc}} \in \mathbb{C}^{B \times k_{\max} \times d_v}$
   - Multiply by learnable complex weight: $\hat{w} = R \cdot \hat{v}_{\text{trunc}}$ where $R \in \mathbb{C}^{k_{\max} \times d_v \times d_v}$
   - Pad back to $N$ modes and apply inverse FFT: $w = \text{IFFT}(\hat{w}_{\text{padded}})$

2. **Residual branch:** $w_{\text{res}} = Wv + b$ (pointwise linear, like a 1×1 convolution)

3. **Combine:** $\text{output} = \sigma(w + w_{\text{res}})$ where $\sigma$ is GELU

**Full FNO:**
- Lift: Linear($d_{\text{in}}$, $d_v$) — project input channels to hidden dimension
- $L$ FNO layers
- Project: Linear($d_v$, $d_{\text{out}}$) — project to output channels

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: FFT Basics (8 points)

**[Coding + Non-coding]**

1. (3 points) **[Coding]** Apply `torch.fft.rfft` to the signal $f(x) = \sin(2\pi x) + 0.5\cos(6\pi x)$ sampled at $N = 64$ points on $[0, 1)$. Print the shape of the FFT output. Why is it $N//2 + 1$ and not $N$?

2. (3 points) **[Coding]** Truncate to the first $k_{\max} = 12$ modes. Apply `torch.fft.irfft` with $n = N$ to reconstruct. Plot the original and reconstructed signal.

3. (2 points) **[Non-coding]** What information is lost when truncating Fourier modes? How does $k_{\max}$ control the trade-off?

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# FFT basics

""" END OF THIS PART """

## Part 2: Spectral Convolution (15 points)

**[Coding]** Implement the core `SpectralConv1d` layer.

This is the Fourier-space multiplication:
1. FFT of input along spatial dimension
2. Truncate to $k_{\max}$ modes
3. Complex multiply with learnable weights $R \in \mathbb{C}^{k_{\max} \times d_{\text{in}} \times d_{\text{out}}}$
4. Pad back and IFFT

**The complex multiplication:** For each Fourier mode $k$, apply a complex matrix multiply:
$$\hat{w}_k = R_k \hat{v}_k$$
where $\hat{v}_k \in \mathbb{C}^{d_{\text{in}}}$ and $R_k \in \mathbb{C}^{d_{\text{in}} \times d_{\text{out}}}$.

In code: use `torch.einsum('bki,kio->bko', x_ft[:, :self.k_max, :], self.weights)`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SpectralConv1d(nn.Module):
    def __init__(self, d_in, d_out, k_max=12):
        """
        Args:
            d_in: input channels
            d_out: output channels
            k_max: number of Fourier modes to keep
        """
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        """
        Args:
            x: (B, N, d_in) — real-valued spatial input
        Returns:
            y: (B, N, d_out) — real-valued spatial output
        """
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: FNO Layer (10 points)

**[Coding]** Combine the spectral convolution with the pointwise residual.

$$\text{FNOLayer}(v) = \text{GELU}(\text{SpectralConv}(v) + Wv)$$

where $W$ is a pointwise linear transform (applied independently at each spatial point).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class FNOLayer(nn.Module):
    def __init__(self, d_v, k_max=12):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, v):
        # v: (B, N, d_v)
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Full FNO Model (10 points)

**[Coding]** Build the complete Fourier Neural Operator.

- Lift: Linear($d_{\text{in}}$, $d_v$)
- 4 FNO layers with $d_v = 64$ and $k_{\max} = 12$
- Project: Linear($d_v$, $d_{\text{out}}$)

For our 1D heat equation: $d_{\text{in}} = 1$ (initial condition value at each point), $d_{\text{out}} = 1$ (solution value at time $T$).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class FNO1d(nn.Module):
    def __init__(self, d_in=1, d_out=1, d_v=64, k_max=12, num_layers=4):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, N, d_in) — input function sampled at N points
        # Returns: (B, N, d_out) — output function at same points
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: Shape Verification (5 points)

**[Coding]** Smoke test the FNO with: $B = 4$, $N = 64$, $d_{\text{in}} = 1$, $d_{\text{out}} = 1$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Smoke test

""" END OF THIS PART """

## Part 6: Generate Training Data (10 points)

**[Coding]** Generate training data from the heat equation $u_t = \alpha u_{xx}$ on $[0, 1]$ with $\alpha = 0.01$.

Use the analytical solution. For random initial conditions of the form:
$$u_0(x) = \sum_{n=1}^{M} a_n \sin(n\pi x)$$

where $a_n \sim \mathcal{N}(0, 1/n^2)$ (higher modes decay).

The solution at time $T$ is:
$$u(T, x) = \sum_{n=1}^{M} a_n e^{-\alpha n^2 \pi^2 T} \sin(n\pi x)$$

Generate:
- 1000 training pairs and 200 test pairs
- $M = 10$ modes, $T = 0.5$
- $N = 64$ spatial grid points

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Generate heat equation data

""" END OF THIS PART """

## Part 7: Train the FNO (10 points)

**[Coding]** Train the FNO to map $u_0 \mapsto u_T$.

- MSE loss
- 200 epochs, Adam lr=1e-3, batch_size=32
- Print train loss and test loss every 50 epochs

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Train FNO

""" END OF THIS PART """

## Part 8: Evaluation and Visualization (8 points)

**[Coding]** Evaluate the trained FNO.

1. (4 points) For 5 test samples, plot the initial condition, the FNO prediction, and the true solution at $T = 0.5$.
2. (4 points) Compute the relative L2 error on the full test set.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Evaluation

""" END OF THIS PART """

## Part 9: PINN vs FNO Comparison (8 points)

**[Non-coding]**

1. (3 points) Compare the PINN approach (Assignment 1) with the FNO approach. What are the advantages and disadvantages of each?

2. (3 points) The FNO was trained on the heat equation with $\alpha = 0.01$. Would it generalize to $\alpha = 0.05$ without retraining? Why or why not? How could you make it generalize?

3. (2 points) The FNO uses a fixed spatial grid of $N = 64$ points. Can it be evaluated at different resolutions (e.g., $N = 128$)? How does the Fourier approach help with this?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 10: Super-Resolution (8 points)

**[Coding]** Test the FNO at a higher resolution than training.

1. (4 points) Generate test data on a grid of $N = 128$ points. Evaluate the FNO by padding the FFT to match the new resolution. Does the FNO generalize to finer grids?

2. (4 points) Compare the error at $N = 128$ vs $N = 64$. Discuss whether the Fourier representation enables resolution-invariance.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Super-resolution test

""" END OF THIS PART """

## Part 11: Computational Complexity (5 points)

**[Non-coding]**

1. (2 points) What is the computational complexity of one FNO layer in terms of $B$, $N$, $d_v$, and $k_{\max}$?

2. (3 points) Compare with: (a) a standard 1D convolution layer with kernel size $K$, and (b) a self-attention layer applied to the $N$ spatial points. For what values of $N$ is the FNO layer more efficient than self-attention?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 12: Extension — Multiple Time Steps (3 points)

**[Non-coding]** The FNO we built predicts $u(T, \cdot)$ from $u(0, \cdot)$ for a single target time $T$. Propose two approaches to extend this to predict $u(t, \cdot)$ for arbitrary $t \in [0, T]$:

1. (1.5 points) Approach A: Autoregressive rollout
2. (1.5 points) Approach B: Time-conditioned FNO

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """